# Feature Attribution using Ranking - v1.0

In [107]:
import sys

!{sys.executable} -m pip install xgboost
!{sys.executable} -m pip install catboost
!{sys.executable} -m pip install nbimporter

In [108]:
import time
import numpy as np
import pandas as pd
import zipfile as zf
import nbimporter

import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb
from catboost import CatBoostClassifier

from scipy.sparse.linalg import eigs

import networkx as nx
import GraphPR as gpr
import PageRank as pr

# Data preprocessing methods

In [4]:
# return one df without nan's on num_cols
# numerical features: fill nan's with median/mean/mode
def pre_proc_fillna_num_fts(df,num_cols,num_type='mean'):
    df_train= df.copy()

    if(num_type=='median'):
        for col in num_cols:
            ft_median= df_train[col].median()
            df_train[col]= df_train[col].fillna(ft_median)
    elif(num_type=='mode'):
        for col in num_cols:
            ft_mode= df_train[col].value_counts().index[0]
            df_train[col]= df_train[col].fillna(ft_mode)
    else:
        for col in num_cols:
            ft_mean= df_train[col].mean()
            df_train[col]= df_train[col].fillna(ft_mean)

    return df_train

In [5]:
# return one df without nan's on cat_cols
# categorical features: fill nan's with mode/mean/median
def pre_proc_fillna_cat_fts(df,cat_cols,cat_type='mode'):
    df_train= df.copy()
    
    if(cat_type!='mode' and type(df_train[cat_cols[0]].value_counts().index[0])!=type('str')):
        if(cat_type=='mean'):
            for col in cat_cols:
                ft_mean= df_train[col].mean()
                df_train[col]= df_train[col].fillna(ft_mean)
        elif(cat_type=='median'):
            for col in cat_cols:
                ft_median= df_train[col].median()
                df_train[col]= df_train[col].fillna(ft_median)
    else:
        for col in cat_cols:
            ft_mode= df_train[col].value_counts().index[0]
            df_train[col]= df_train[col].fillna(ft_mode)

    return df_train

In [6]:
# n_cols refers only to df's columns with numerical values
def normalize_selected_cols(df, n_cols):
    result= df.copy()
    
    for col in n_cols:
        max_value= df[col].max()
        min_value= df[col].min()
        result[col]= (df[col]- min_value)/ (max_value - min_value)
        
    return result

# Feature Attribution using Ranking methods

In [7]:
# return a DataFrame with replace values (mean/median/mode/none to categorical and numeric) to fill train cols. df is post-processed (cat_cols encoded)
def replace_values(df,num_cols,num_type='mean',cat_type='none'):
    cat_values= None
    num_values= None
    
    if (cat_type=='mean'):
        cat_values= df.mean(axis=0).to_frame().T
    elif (cat_type=='median'):
        cat_values= df.median(axis=0).to_frame().T
    elif (cat_type=='mode'):
        cat_values= df.mode(axis=0)
    
    if (num_type=='mode'):
        num_values= df.mode(axis=0)
    elif (num_type=='median'):
        num_values= df.median(axis=0).to_frame().T
    elif (num_type=='mean'):
        num_values= df.mean(axis=0).to_frame().T

    if(cat_type!='none'):
        cat_values[num_cols]= num_values[num_cols]
        return cat_values
    
    return num_values

In [8]:
# train the ML model and return its mean accuracy after n_train runs
def train_model_get_acc_mean(model, x_trn, x_tst, y_trn, y_tst, n_train):
    trainings= []

    for i in range(n_train):

        model.fit(x_trn, y_trn)
        acc= sklearn.metrics.accuracy_score(y_tst, model.predict(x_tst))

        trainings.append(acc)

    return np.mean(trainings)

In [9]:
# re-training is needed because machine learning models typically assume that the train and the test data comes from a similar distribution 
# (Hooker et al., 2018)
# here we return p(x|i) and p(x|ij)
def remove_and_retrain_v1(model, replace_ft_vals, x_trn, x_tst, y_trn, y_tst, n_train, verbose=False):
    acc_no_i= []
    acc_no_ij= []

    n_fts= len(x_trn.columns)

    start= time.time()

    for i in range(n_fts):
        acc_row= []

        # replace the i-th ft with its respective mode/mean to "remove" it. train the ML model and get the mean accuracy
        train_copy_no_i= x_trn.copy()
        train_copy_no_i.loc[:,train_copy_no_i.columns[i]]= replace_ft_vals.iloc[0,i]

        acc_no_i.append(train_model_get_acc_mean(model, train_copy_no_i, x_tst, y_trn, y_tst, n_train))

        for j in range(n_fts):

            if (i!= j):
                # replace the j-th ft with its respective mode/mean to "remove" it. here, we "remove" the i-th and the j-th ft
                # train the ML model and get the mean accuracy
                train_copy_no_ij= train_copy_no_i.copy()
                train_copy_no_ij.loc[:,train_copy_no_ij.columns[j]]= replace_ft_vals.iloc[0,j]

                acc_row.append(train_model_get_acc_mean(model, train_copy_no_ij, x_tst, y_trn, y_tst, n_train))
            else:
                acc_row.append(0)

        acc_no_ij.append(acc_row)

    end= time.time()
    
    if (verbose==True):
        print("--- %s seconds ---" % np.round((end- start), 2))

    return acc_no_i, acc_no_ij

In [10]:
# here we return | p(x|ij) - p(x|i) |
def get_p_matrix_v1(n_fts, acc_no_i, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= abs(pij- pi)
                
    return p_matrix

In [11]:
# here we return | p(x|ij) - p(x|j) |
def get_p_matrix_v2(n_fts, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))

    for i in range(n_fts):
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= abs(pij- pj)
                
    return p_matrix

In [12]:
# here we return (| p(x|ij) - p(x|j) | + | p(x|i) - p(x) |) / 2
def get_p_matrix_v3(n_fts, acc_all, acc_no_i, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))
    p= acc_all

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= (abs(pij- pj)+ abs(pi- p))/ 2
                
    return p_matrix

In [13]:
# here we return (| p(x|ij) - p(x|i) | + | p(x|j) - p(x) |) / 2
def get_p_matrix_v4(n_fts, acc_all, acc_no_i, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))
    p= acc_all

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= (abs(pij- pi)+ abs(pj- p))/ 2
                
    return p_matrix

In [14]:
# the stationary distribution is the fraction of time that the system spends in each state as the number of samples approaches infinity
# it looks like there's not a built-in method to find the stationary distribution

# converts a matrix to a row stochastic matrix - a real square matrix, with each row summing to 1
def to_row_stochastic_matrix(M):
    result= M
    
    for row in result:
        n= sum(row)
        if n> 0:
            row[:]= [f/sum(row) for f in row]
    
    return result

In [15]:
# the stationary distribution - analytical solution
# return 1D array
def stationary_dist_v1(stochastic_matrix):
    
    size_A= stochastic_matrix.shape[1]
    ones= [1]* size_A

    A= np.append(np.transpose(stochastic_matrix)- np.identity(size_A),[ones],axis=0)

    v= np.zeros(size_A+ 1)
    v[size_A]= 1
    v= np.transpose(v)

    stationary= np.linalg.solve(np.transpose(A).dot(A), np.transpose(A).dot(v))

    return stationary

In [16]:
# the stationary distribution - another analytical solution
# return 2D array
def stationary_dist_v2(stochastic_matrix):
    # we have to transpose so that Markov transitions correspond to right multiplying by a column vector
    eigval, eigvec= eigs(stochastic_matrix.T, k=1, which='LM')
    stationary= eigvec/ eigvec.sum()

    # eigs finds complex eigenvalues and eigenvectors, so you'll want the real part.
    stationary= stationary.real

    return stationary

In [17]:
# return a sorted DataFrame with Features and Importances - OHE compacted, that is, dataset's original features
def ft_importance_df(importances, ft_names, replace_list):
    fti= pd.Series(importances, index=ft_names).sort_values(ascending=False).to_frame().reset_index()
    fti= fti.rename(columns= {'index':'Feature',0:'Importance'}, inplace=False)
    fti['Feature'].replace(replace_list, inplace=True)
    fti= fti.groupby(['Feature']).sum().sort_values('Importance', ascending=False).reset_index()
    
    return fti

In [62]:
import math
from sklearn.neighbors import NearestNeighbors

# find the test_size nearest neighbors from a target_X instance
# return train, test, labels_train, labels_test datsets based on knn, with target_X and target_Y into testX and testY
def knn_train_test_split(df_X, df_Y, target_X, target_Y, test_size= 0.2):
    
    m_ins= df_X.shape[0]

    neighbors= math.floor(test_size* m_ins)

    knn= NearestNeighbors(n_neighbors= neighbors)
    knn.fit(df_X)

    knn_index= knn.kneighbors(target_X, return_distance=False)
    
    trainX= df_X.drop(df_X.index[knn_index[0]])
    trainY= df_Y.drop(df_Y.index[knn_index[0]])

    testX= df_X.loc[df_X.index[knn_index[0]]]
    testY= df_Y.loc[df_Y.index[knn_index[0]]]

    testX= pd.concat([target_X, testX])
    testY= pd.concat([target_Y, testY])
    
    return trainX, testX, trainY, testY

In [76]:
from sklearn.model_selection import KFold, StratifiedKFold

# df_X and df_Y doesn't contain target_X and target_Y
def knnfold_remove_and_retrain(model, df_X, df_Y, target_X, target_Y, numeric_columns, num_type='mean', cat_type='none', test_size=0.01):

    m_ins= df_X.shape[0]
    n_fts= df_X.shape[1]
    
    k_viz= math.floor(test_size* m_ins)
    k_viz_aux= math.floor(np.sqrt(m_ins))
    
    if (k_viz< k_viz_aux):
        k_viz= k_viz_aux
        
    # here, we generate a neighborhood of the target instance, our test set
    x_train, x_test, y_train, y_test= knn_train_test_split(df_X, df_Y, target_X, target_Y, 
                                                                         test_size= (k_viz/ m_ins))    
    K_folds= math.floor(np.sqrt(n_fts))

    if (K_folds< 5):
        K_folds= 5    

    skf= StratifiedKFold(n_splits= K_folds, random_state=1234, shuffle=True)    
        
    acc_all= []
    acc_no_i= np.zeros(n_fts)
    acc_no_ij= np.zeros((n_fts,n_fts))
    
    for train_index, test_index in skf.split(x_train, y_train):
        
        x_t_fold= x_train.iloc[train_index]
        y_t_fold= y_train.iloc[train_index]
        
        acc_all.append(train_model_get_acc_mean(model, x_t_fold, x_test, y_t_fold, y_test, 1))
        
        replace_ft= replace_values(x_t_fold, numeric_columns, num_type=num_type, cat_type=cat_type)
        
        aux_acc_no_i, aux_acc_no_ij= remove_and_retrain_v1(model, replace_ft, x_t_fold, x_test, 
                                                           y_t_fold, y_test, 1)
        
        acc_no_i += aux_acc_no_i
        acc_no_ij += aux_acc_no_ij

    
    acc_no_i /= K_folds
    acc_no_ij /= K_folds

    mean_acc_all= np.mean(acc_all)
    
    return mean_acc_all, acc_no_i, acc_no_ij

# Feature Attribution using PageRank methods

In [109]:
def run_lib_pr(graph_matrix, ft_names):
    
    num_fts= graph_matrix.shape[0]
    
    D= nx.DiGraph()

    for i in range(num_fts):
        for j in range(num_fts):
            if (i!= j):
                D.add_weighted_edges_from([(ft_names[i],ft_names[j],graph_matrix[i,j])])
                
    pRank= pd.Series(nx.pagerank(D, max_iter=100, alpha=0.85, tol=1.0e-6))

    return pRank.sort_values(ascending=False)

# --- Tests using the Titanic dataset ---

# Data loading and preprocessing

In [17]:
!kaggle competitions download -c titanic

/bin/bash: kaggle: command not found


In [21]:
ds= zf.ZipFile('datasets/titanic.zip')

train_data= pd.read_csv(ds.open('train.csv'))
test_data= pd.read_csv(ds.open('test.csv'))

train_data.shape, test_data.shape

((891, 12), (418, 11))

In [22]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [23]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [24]:
X_all= pd.concat([train_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']],
                   test_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]]).set_index('PassengerId')

y_train= train_data[['PassengerId','Survived']].set_index('PassengerId')['Survived']

X_all

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.2500,S
2,1,female,38.0,1,0,71.2833,C
3,3,female,26.0,0,0,7.9250,S
4,1,female,35.0,1,0,53.1000,S
5,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...
1305,3,male,NaN,0,0,8.0500,S
1306,1,female,39.0,0,0,108.9000,C
1307,3,male,38.5,0,0,7.2500,S


In [25]:
numeric_columns= ['Age','SibSp','Parch','Fare']
categor_columns= list(filter(lambda x:x not in numeric_columns,X_all.columns))

X_train= X_all.iloc[:len(train_data),:].copy()
X_test= X_all.iloc[len(train_data):].copy()

In [26]:
# in this case we'll only use X_train df because X_test is not labeled

In [27]:
X_train= pre_proc_fillna_num_fts(X_train,numeric_columns,num_type='median')

X_train= pre_proc_fillna_cat_fts(X_train,categor_columns,cat_type='mode')

X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.2500,S
2,1,female,38.0,1,0,71.2833,C
3,3,female,26.0,0,0,7.9250,S
4,1,female,35.0,1,0,53.1000,S
5,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...
887,2,male,27.0,0,0,13.0000,S
888,1,female,19.0,0,0,30.0000,S
889,3,female,28.0,1,2,23.4500,S


In [28]:
# one-hot encoding the qualitative features
X_train_ohe= pd.get_dummies(X_train,columns=categor_columns)

X_train_ohe.head()

,Age,SibSp,Parch,Fare,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
PassengerId,,,,,,,,,,,,
1,22.0,1,0,7.2500,0,0,1,0,1,0,0,1
2,38.0,1,0,71.2833,1,0,0,1,0,1,0,0
3,26.0,0,0,7.9250,0,0,1,1,0,0,0,1
4,35.0,1,0,53.1000,1,0,0,1,0,0,0,1
5,35.0,0,0,8.0500,0,0,1,0,1,0,0,1


In [29]:
y_train.head()

PassengerId
1    0
2    1
3    1
4    1
5    0
Name: Survived, dtype: int64

In [30]:
# normalize the numeric columns of dataframe with each value between 0 and 1
X_train= normalize_selected_cols(X_train, numeric_columns)
X_train_ohe= normalize_selected_cols(X_train_ohe, numeric_columns)

# ML model setup

In [31]:
train, test, labels_train, labels_test= train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)
rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2)

#train, test, labels_train, labels_test= train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)
#xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)

#train, test, labels_train, labels_test= train_test_split(X_train,y_train,train_size=0.80,random_state=1234)
#cat_fts= [train.columns.to_list().index(col) for col in categor_columns]
#ctb_model= CatBoostClassifier(cat_features=cat_fts,silent=True)

# --- Run global modeling ---

In [80]:
# re-training can result in slightly different models, it is essential to repeat the training process multiple times to ensure that the variance in accuracy is low 
# (Hooker et al., 2018)
repeat_train= 1

num_fts= len(train.columns)

In [35]:
# train the ML model and get the mean accuracy using the entire feature set
acc_all_fts= train_model_get_acc_mean(rf, train, test, labels_train, labels_test, repeat_train)

acc_all_fts

0.8212290502793296

In [36]:
# values to "remove" and retrain
replace_ft= replace_values(train,numeric_columns,num_type='mean',cat_type='median')

replace_ft

,Age,SibSp,Parch,Fare,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0.362142,0.064782,0.064841,0.064072,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0


In [33]:
# train the ML model and get the mean accuracy removing and retraining columns from the feature set
acc_no_i, acc_no_ij= remove_and_retrain_v1(rf, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [34]:
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

[0.793 0.816 0.821 0.827 0.827 0.821 0.827 0.821 0.821 0.821 0.821 0.821]
-----------------------------
[[0.    0.804 0.832 0.81  0.804 0.793 0.793 0.799 0.804 0.81  0.799 0.793]
 [0.804 0.    0.816 0.799 0.821 0.821 0.821 0.81  0.81  0.816 0.816 0.81 ]
 [0.832 0.821 0.    0.827 0.827 0.827 0.827 0.816 0.821 0.838 0.816 0.821]
 [0.804 0.804 0.821 0.    0.832 0.832 0.827 0.832 0.827 0.827 0.832 0.827]
 [0.799 0.821 0.827 0.838 0.    0.827 0.827 0.821 0.821 0.827 0.827 0.821]
 [0.799 0.821 0.816 0.832 0.827 0.    0.793 0.821 0.821 0.821 0.821 0.816]
 [0.793 0.816 0.827 0.832 0.827 0.793 0.    0.821 0.821 0.821 0.821 0.821]
 [0.804 0.816 0.816 0.827 0.821 0.821 0.816 0.    0.704 0.821 0.821 0.816]
 [0.793 0.81  0.81  0.827 0.821 0.821 0.821 0.682 0.    0.821 0.816 0.81 ]
 [0.804 0.81  0.838 0.821 0.827 0.827 0.821 0.816 0.821 0.    0.816 0.81 ]
 [0.799 0.821 0.827 0.832 0.821 0.821 0.821 0.821 0.821 0.816 0.    0.804]
 [0.799 0.821 0.832 0.832 0.816 0.816 0.821 0.81  0.816 0.821 0.816 0. 

In [35]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

#print(p_matrix1)
print(np.around(p_matrix1, decimals=3))
print("-----------------------------")
#print(p_matrix2)
print(np.around(p_matrix2, decimals=3))
print("-----------------------------")
#print(p_matrix3)
print(np.around(p_matrix3, decimals=3))
print("-----------------------------")
#print(p_matrix4)
print(np.around(p_matrix4, decimals=3))

[[0.    0.011 0.039 0.017 0.011 0.    0.    0.006 0.011 0.017 0.006 0.   ]
 [0.011 0.    0.    0.017 0.006 0.006 0.006 0.006 0.006 0.    0.    0.006]
 [0.011 0.    0.    0.006 0.006 0.006 0.006 0.006 0.    0.017 0.006 0.   ]
 [0.022 0.022 0.006 0.    0.006 0.006 0.    0.006 0.    0.    0.006 0.   ]
 [0.028 0.006 0.    0.011 0.    0.    0.    0.006 0.006 0.    0.    0.006]
 [0.022 0.    0.006 0.011 0.006 0.    0.028 0.    0.    0.    0.    0.006]
 [0.034 0.011 0.    0.006 0.    0.034 0.    0.006 0.006 0.006 0.006 0.006]
 [0.017 0.006 0.006 0.006 0.    0.    0.006 0.    0.117 0.    0.    0.006]
 [0.028 0.011 0.011 0.006 0.    0.    0.    0.14  0.    0.    0.006 0.011]
 [0.017 0.011 0.017 0.    0.006 0.006 0.    0.006 0.    0.    0.006 0.011]
 [0.022 0.    0.006 0.011 0.    0.    0.    0.    0.    0.006 0.    0.017]
 [0.022 0.    0.011 0.011 0.006 0.006 0.    0.011 0.006 0.    0.006 0.   ]]
-----------------------------
[[0.    0.011 0.011 0.017 0.022 0.028 0.034 0.022 0.017 0.011 0.022 0

In [36]:
# finding the stationary distribution
st_matrix1= np.asarray(p_matrix1)
st_matrix2= np.asarray(p_matrix2)
st_matrix3= np.asarray(p_matrix3)
st_matrix4= np.asarray(p_matrix4)

# now convert to right stochastic matrix - a real square matrix, with each row summing to 1
st_matrix1= to_row_stochastic_matrix(st_matrix1)
st_matrix2= to_row_stochastic_matrix(st_matrix2)
st_matrix3= to_row_stochastic_matrix(st_matrix3)
st_matrix4= to_row_stochastic_matrix(st_matrix4)

print(st_matrix1.sum())
print(st_matrix2.sum())
print(st_matrix3.sum())
print(st_matrix4.sum())

12.0
12.0
12.0
12.0


In [37]:
# get the stationary distribution
stationary_d1= stationary_dist_v1(st_matrix1)
stationary_d2= stationary_dist_v1(st_matrix2)
stationary_d3= stationary_dist_v1(st_matrix3)
stationary_d4= stationary_dist_v1(st_matrix4)

print(stationary_d1)
print("-----------------------------")
print(stationary_d2)
print("-----------------------------")
print(stationary_d3)
print("-----------------------------")
print(stationary_d4)

[0.18002355 0.07487006 0.10504361 0.0947893  0.0514202  0.04210641
 0.03612971 0.13734167 0.13324574 0.05965845 0.03834663 0.04702468]
-----------------------------
[0.09688042 0.05350664 0.07679715 0.07196842 0.0418801  0.11985257
 0.10998994 0.14484344 0.13214301 0.03923308 0.04584324 0.067062  ]
-----------------------------
[0.0982633  0.0553632  0.07784017 0.05804254 0.04990515 0.0690063
 0.08915894 0.17159756 0.16051876 0.04758638 0.04997886 0.07273883]
-----------------------------
[0.26717294 0.09732554 0.09698515 0.11084627 0.07909679 0.0311486
 0.05863646 0.07750169 0.07673875 0.05009644 0.02787507 0.0265763 ]


In [38]:
rp_list= {'Sex_male':'Sex','Sex_female':'Sex','Pclass_1':'Pclass','Pclass_2':'Pclass','Pclass_3':'Pclass',
          'Embarked_S':'Embarked','Embarked_Q':'Embarked','Embarked_C':'Embarked'}

In [39]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d1, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.180024
Sex_female    0.137342
Sex_male      0.133246
Parch         0.105044
Fare          0.094789
SibSp         0.074870
Embarked_C    0.059658
Pclass_1      0.051420
Embarked_S    0.047025
Pclass_2      0.042106
Embarked_Q    0.038347
Pclass_3      0.036130
dtype: float64

In [40]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.270587
1,Age,0.180024
2,Embarked,0.145030
3,Pclass,0.129656
4,Parch,0.105044
5,Fare,0.094789
6,SibSp,0.074870


In [41]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d2, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Sex_female    0.144843
Sex_male      0.132143
Pclass_2      0.119853
Pclass_3      0.109990
Age           0.096880
Parch         0.076797
Fare          0.071968
Embarked_S    0.067062
SibSp         0.053507
Embarked_Q    0.045843
Pclass_1      0.041880
Embarked_C    0.039233
dtype: float64

In [42]:
ft_importance_df(stationary_d2,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.276986
1,Pclass,0.271723
2,Embarked,0.152138
3,Age,0.096880
4,Parch,0.076797
5,Fare,0.071968
6,SibSp,0.053507


In [43]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d3, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Sex_female    0.171598
Sex_male      0.160519
Age           0.098263
Pclass_3      0.089159
Parch         0.077840
Embarked_S    0.072739
Pclass_2      0.069006
Fare          0.058043
SibSp         0.055363
Embarked_Q    0.049979
Pclass_1      0.049905
Embarked_C    0.047586
dtype: float64

In [44]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.270587
1,Age,0.180024
2,Embarked,0.145030
3,Pclass,0.129656
4,Parch,0.105044
5,Fare,0.094789
6,SibSp,0.074870


In [45]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d4, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.267173
Fare          0.110846
SibSp         0.097326
Parch         0.096985
Pclass_1      0.079097
Sex_female    0.077502
Sex_male      0.076739
Pclass_3      0.058636
Embarked_C    0.050096
Pclass_2      0.031149
Embarked_Q    0.027875
Embarked_S    0.026576
dtype: float64

In [46]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.270587
1,Age,0.180024
2,Embarked,0.145030
3,Pclass,0.129656
4,Parch,0.105044
5,Fare,0.094789
6,SibSp,0.074870


# --- Run local modeling ---

In [86]:
X_to_split= X_train_ohe.copy()
Y_to_split= y_train.copy()

In [87]:
target_index= 1

target_instance= X_to_split.loc[X_to_split.index== target_index]
target_label= Y_to_split.loc[Y_to_split.index== target_index]

X_no_targt= X_to_split.drop(target_instance.index)
Y_no_targt= Y_to_split.drop(index= target_index)

In [88]:
Y_no_targt.head()

PassengerId
2    1
3    1
4    1
5    0
6    0
Name: Survived, dtype: int64

In [89]:
num_fts= len(X_no_targt.columns)

acc_all_fts, acc_no_i, acc_no_ij= knnfold_remove_and_retrain(rf, X_no_targt, Y_no_targt, target_instance, 
                                                              target_label, numeric_columns, num_type='mean', 
                                                              cat_type='median', test_size= 0.2)

In [90]:
print(acc_all_fts)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

0.8826815642458101
-----------------------------
[0.87  0.88  0.882 0.396 0.883 0.88  0.882 0.882 0.883 0.882 0.88  0.883]
-----------------------------
[[0.    0.873 0.829 0.875 0.872 0.868 0.869 0.872 0.87  0.874 0.868 0.868]
 [0.873 0.    0.877 0.356 0.883 0.88  0.882 0.88  0.88  0.88  0.88  0.88 ]
 [0.83  0.877 0.    0.316 0.882 0.88  0.88  0.883 0.883 0.882 0.88  0.882]
 [0.873 0.41  0.304 0.    0.241 0.448 0.403 0.334 0.353 0.404 0.47  0.468]
 [0.873 0.883 0.883 0.293 0.    0.883 0.878 0.883 0.883 0.882 0.883 0.883]
 [0.868 0.88  0.88  0.451 0.883 0.    0.876 0.882 0.88  0.882 0.88  0.883]
 [0.868 0.882 0.882 0.448 0.879 0.876 0.    0.883 0.882 0.883 0.88  0.882]
 [0.87  0.88  0.883 0.268 0.883 0.88  0.882 0.    0.664 0.883 0.882 0.883]
 [0.869 0.88  0.882 0.304 0.883 0.882 0.883 0.669 0.    0.883 0.882 0.883]
 [0.873 0.88  0.88  0.308 0.882 0.883 0.883 0.883 0.883 0.    0.883 0.802]
 [0.868 0.88  0.883 0.421 0.883 0.88  0.88  0.883 0.883 0.883 0.    0.877]
 [0.869 0.882 0.883 0.

In [91]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

#print(p_matrix1)
print(np.around(p_matrix1, decimals=3))
print("-----------------------------")
#print(p_matrix2)
print(np.around(p_matrix2, decimals=3))
print("-----------------------------")
#print(p_matrix3)
print(np.around(p_matrix3, decimals=3))
print("-----------------------------")
#print(p_matrix4)
print(np.around(p_matrix4, decimals=3))

[[0.    0.002 0.041 0.004 0.001 0.002 0.001 0.001 0.    0.003 0.002 0.002]
 [0.008 0.    0.003 0.524 0.002 0.    0.001 0.    0.    0.    0.    0.   ]
 [0.051 0.004 0.    0.565 0.    0.001 0.001 0.001 0.001 0.    0.001 0.   ]
 [0.477 0.015 0.092 0.    0.154 0.053 0.008 0.061 0.042 0.009 0.075 0.073]
 [0.01  0.    0.    0.59  0.    0.    0.004 0.    0.    0.001 0.    0.   ]
 [0.012 0.    0.    0.429 0.002 0.    0.004 0.001 0.    0.001 0.    0.002]
 [0.013 0.    0.    0.434 0.002 0.006 0.    0.001 0.    0.001 0.001 0.   ]
 [0.011 0.001 0.001 0.613 0.001 0.001 0.    0.    0.218 0.001 0.    0.001]
 [0.013 0.002 0.001 0.579 0.    0.001 0.    0.213 0.    0.    0.001 0.   ]
 [0.009 0.001 0.001 0.573 0.    0.001 0.001 0.001 0.001 0.    0.001 0.079]
 [0.012 0.    0.002 0.459 0.002 0.    0.    0.002 0.002 0.002 0.    0.003]
 [0.013 0.001 0.    0.413 0.    0.002 0.    0.    0.    0.076 0.006 0.   ]]
-----------------------------
[[0.    0.008 0.053 0.479 0.011 0.012 0.012 0.01  0.012 0.008 0.012 0

In [92]:
# finding the stationary distribution
st_matrix1= np.asarray(p_matrix1)
st_matrix2= np.asarray(p_matrix2)
st_matrix3= np.asarray(p_matrix3)
st_matrix4= np.asarray(p_matrix4)

# now convert to right stochastic matrix - a real square matrix, with each row summing to 1
st_matrix1= to_row_stochastic_matrix(st_matrix1)
st_matrix2= to_row_stochastic_matrix(st_matrix2)
st_matrix3= to_row_stochastic_matrix(st_matrix3)
st_matrix4= to_row_stochastic_matrix(st_matrix4)

# get the stationary distribution
stationary_d1= stationary_dist_v1(st_matrix1)
stationary_d2= stationary_dist_v1(st_matrix2)
stationary_d3= stationary_dist_v1(st_matrix3)
stationary_d4= stationary_dist_v1(st_matrix4)

print(stationary_d1)
print("-----------------------------")
print(stationary_d2)
print("-----------------------------")
print(stationary_d3)
print("-----------------------------")
print(stationary_d4)

[0.19288547 0.01371236 0.16360422 0.38674766 0.06029593 0.02685897
 0.00742878 0.0329755  0.02446565 0.01957815 0.03514872 0.03629857]
-----------------------------
[0.02440143 0.03666561 0.04901844 0.34937147 0.04830312 0.03572071
 0.04113406 0.12478816 0.11918876 0.06958404 0.03331239 0.06851181]
-----------------------------
[0.0415524  0.03862461 0.04722655 0.32801993 0.04484345 0.03922806
 0.04262696 0.12108639 0.11469663 0.07300111 0.03765654 0.07143737]
-----------------------------
[0.2227829  0.01004667 0.0579107  0.47343343 0.06814272 0.02651304
 0.00557448 0.03261093 0.02396204 0.00918233 0.03633204 0.03350872]


In [93]:
rp_list= {'Sex_male':'Sex','Sex_female':'Sex','Pclass_1':'Pclass','Pclass_2':'Pclass','Pclass_3':'Pclass',
          'Embarked_S':'Embarked','Embarked_Q':'Embarked','Embarked_C':'Embarked'}

In [94]:
# TARGET INSTANCE TO EXPLAIN ITS FEATURES

X_all.loc[X_all.index== target_index]

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.25,S


In [95]:
target_label

PassengerId
1    0
Name: Survived, dtype: int64

In [96]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d1, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Fare          0.386748
Age           0.192885
Parch         0.163604
Pclass_1      0.060296
Embarked_S    0.036299
Embarked_Q    0.035149
Sex_female    0.032976
Pclass_2      0.026859
Sex_male      0.024466
Embarked_C    0.019578
SibSp         0.013712
Pclass_3      0.007429
dtype: float64

In [97]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Fare,0.386748
1,Age,0.192885
2,Parch,0.163604
3,Pclass,0.094584
4,Embarked,0.091025
5,Sex,0.057441
6,SibSp,0.013712


In [98]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d2, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Fare          0.349371
Sex_female    0.124788
Sex_male      0.119189
Embarked_C    0.069584
Embarked_S    0.068512
Parch         0.049018
Pclass_1      0.048303
Pclass_3      0.041134
SibSp         0.036666
Pclass_2      0.035721
Embarked_Q    0.033312
Age           0.024401
dtype: float64

In [99]:
ft_importance_df(stationary_d2,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Fare,0.349371
1,Sex,0.243977
2,Embarked,0.171408
3,Pclass,0.125158
4,Parch,0.049018
5,SibSp,0.036666
6,Age,0.024401


In [100]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d3, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Fare          0.328020
Sex_female    0.121086
Sex_male      0.114697
Embarked_C    0.073001
Embarked_S    0.071437
Parch         0.047227
Pclass_1      0.044843
Pclass_3      0.042627
Age           0.041552
Pclass_2      0.039228
SibSp         0.038625
Embarked_Q    0.037657
dtype: float64

In [101]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Fare,0.386748
1,Age,0.192885
2,Parch,0.163604
3,Pclass,0.094584
4,Embarked,0.091025
5,Sex,0.057441
6,SibSp,0.013712


In [102]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d4, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Fare          0.473433
Age           0.222783
Pclass_1      0.068143
Parch         0.057911
Embarked_Q    0.036332
Embarked_S    0.033509
Sex_female    0.032611
Pclass_2      0.026513
Sex_male      0.023962
SibSp         0.010047
Embarked_C    0.009182
Pclass_3      0.005574
dtype: float64

In [103]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Fare,0.386748
1,Age,0.192885
2,Parch,0.163604
3,Pclass,0.094584
4,Embarked,0.091025
5,Sex,0.057441
6,SibSp,0.013712


# --- Run PageRank approach ---

In [111]:
X_to_split= X_train_ohe.copy()
Y_to_split= y_train.copy()

target_index= 1

target_instance= X_to_split.loc[X_to_split.index== target_index]
target_label= Y_to_split.loc[Y_to_split.index== target_index]

X_no_targt= X_to_split.drop(target_instance.index)
Y_no_targt= Y_to_split.drop(index= target_index)

In [112]:
num_fts= len(X_no_targt.columns)

acc_all_fts, acc_no_i, acc_no_ij= knnfold_remove_and_retrain(rf, X_no_targt, Y_no_targt, target_instance, 
                                                              target_label, numeric_columns, num_type='mean', 
                                                              cat_type='median', test_size= 0.2)

In [113]:
print(acc_all_fts)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

0.8826815642458101
-----------------------------
[0.873 0.88  0.88  0.268 0.883 0.882 0.882 0.883 0.883 0.883 0.883 0.883]
-----------------------------
[[0.    0.876 0.828 0.875 0.873 0.869 0.869 0.869 0.872 0.872 0.868 0.87 ]
 [0.879 0.    0.877 0.415 0.883 0.88  0.88  0.88  0.88  0.88  0.88  0.88 ]
 [0.829 0.877 0.    0.383 0.883 0.88  0.882 0.88  0.88  0.883 0.88  0.883]
 [0.875 0.41  0.381 0.    0.231 0.385 0.389 0.266 0.316 0.301 0.469 0.465]
 [0.873 0.883 0.883 0.229 0.    0.883 0.878 0.883 0.883 0.883 0.883 0.883]
 [0.869 0.88  0.88  0.42  0.883 0.    0.872 0.883 0.882 0.88  0.88  0.88 ]
 [0.868 0.88  0.88  0.41  0.879 0.874 0.    0.883 0.883 0.882 0.882 0.883]
 [0.868 0.882 0.88  0.333 0.883 0.882 0.882 0.    0.661 0.882 0.882 0.883]
 [0.872 0.88  0.882 0.315 0.883 0.882 0.882 0.669 0.    0.882 0.882 0.883]
 [0.873 0.88  0.88  0.317 0.882 0.883 0.883 0.882 0.883 0.    0.883 0.804]
 [0.868 0.88  0.882 0.468 0.883 0.88  0.88  0.882 0.882 0.883 0.    0.876]
 [0.869 0.88  0.883 0.

In [116]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# finding the stationary distribution
st_matrix1= np.asarray(p_matrix1)
st_matrix2= np.asarray(p_matrix2)
st_matrix3= np.asarray(p_matrix3)
st_matrix4= np.asarray(p_matrix4)

# now convert to right stochastic matrix - a real square matrix, with each row summing to 1
st_matrix1= to_row_stochastic_matrix(st_matrix1)
st_matrix2= to_row_stochastic_matrix(st_matrix2)
st_matrix3= to_row_stochastic_matrix(st_matrix3)
st_matrix4= to_row_stochastic_matrix(st_matrix4)

In [117]:
run_lib_pr(st_matrix1, X_train_ohe.columns)

Fare          0.374274
Age           0.147459
Parch         0.116340
Embarked_Q    0.061006
Embarked_S    0.059565
SibSp         0.047925
Pclass_3      0.043923
Pclass_2      0.042561
Sex_male      0.030408
Embarked_C    0.029668
Sex_female    0.026238
Pclass_1      0.020634
dtype: float64

In [118]:
run_lib_pr(st_matrix2, X_train_ohe.columns)

Fare          0.319953
Sex_female    0.136490
Sex_male      0.131121
Embarked_S    0.066484
Embarked_C    0.059185
Pclass_1      0.048055
Pclass_3      0.044601
Parch         0.043869
Pclass_2      0.042862
SibSp         0.040280
Embarked_Q    0.038308
Age           0.028792
dtype: float64

In [119]:
run_lib_pr(st_matrix3, X_train_ohe.columns)

Fare          0.315989
Sex_female    0.129803
Sex_male      0.125489
Embarked_S    0.069010
Embarked_C    0.057861
Pclass_3      0.044947
Parch         0.044346
Pclass_1      0.044207
Pclass_2      0.043260
Age           0.042420
Embarked_Q    0.041500
SibSp         0.041167
dtype: float64

In [120]:
run_lib_pr(st_matrix4, X_train_ohe.columns)

Fare          0.445813
Age           0.163044
Embarked_Q    0.060422
Embarked_S    0.060272
Parch         0.049497
SibSp         0.048091
Pclass_3      0.042588
Pclass_2      0.041540
Sex_male      0.026223
Embarked_C    0.024014
Pclass_1      0.021340
Sex_female    0.017156
dtype: float64

In [130]:
# using my PR to compare results
def run_my_pr(graph_matrix, ft_names):
    
    file_path= 'datasets/FAR_data.txt'
    
    num_fts= graph_matrix.shape[0]

    f= open(file_path, 'w')

    for i in range(num_fts):
        for j in range(num_fts):
            line= (str(int(i)) + ',' + str(int(j)) + ',' + str(graph_matrix[i,j]) + '\n')
            f.write(line)

    f.close()

    graph= gpr.init_graph(file_path)

    myPRank= pd.Series(pr.run_PageRank(graph, iteration= 100, damping_factor= 0.95, tolerance= 1.0e-6))
    myPRank= myPRank.sort_values(ascending=False)
    
    ids_names= ft_names[(myPRank.index).astype(int)]

    myPRank.index= ids_names

    return myPRank

In [132]:
run_my_pr(st_matrix1, X_train_ohe.columns)

Fare          0.322565
Age           0.155406
Parch         0.070471
SibSp         0.068480
Sex_female    0.054198
Embarked_C    0.050457
Embarked_Q    0.047883
Embarked_S    0.047429
Pclass_2      0.046716
Pclass_3      0.046460
Sex_male      0.045618
Pclass_1      0.044316
dtype: float64

In [133]:
run_my_pr(st_matrix2, X_train_ohe.columns)

Fare          0.305249
Sex_female    0.091676
Parch         0.077100
SibSp         0.075994
Age           0.073028
Embarked_C    0.061879
Sex_male      0.055922
Pclass_2      0.053014
Embarked_S    0.052519
Pclass_1      0.051859
Pclass_3      0.051142
Embarked_Q    0.050618
dtype: float64

In [134]:
run_my_pr(st_matrix3, X_train_ohe.columns)

Fare          0.296607
Sex_female    0.090584
Age           0.085234
SibSp         0.076131
Parch         0.075831
Embarked_C    0.061399
Sex_male      0.055483
Pclass_2      0.052895
Embarked_S    0.052532
Pclass_1      0.051726
Pclass_3      0.050933
Embarked_Q    0.050645
dtype: float64

In [135]:
run_my_pr(st_matrix4, X_train_ohe.columns)

Fare          0.340196
Age           0.157398
SibSp         0.069066
Parch         0.064516
Sex_female    0.048795
Embarked_Q    0.046798
Embarked_S    0.046707
Embarked_C    0.046450
Pclass_2      0.045816
Pclass_3      0.045704
Sex_male      0.044618
Pclass_1      0.043935
dtype: float64

# --- Using knn to reduce the dataset ---

In [136]:
target_index= 1

target_instance= X_to_split.loc[X_to_split.index== target_index]
target_label= Y_to_split.loc[Y_to_split.index== target_index]

X_no_targt= X_to_split.drop(target_instance.index)
Y_no_targt= Y_to_split.drop(index= target_index)

In [137]:
train, test, labels_train, labels_test= knn_train_test_split(X_no_targt,Y_no_targt,
                                                            target_instance,target_label,test_size=0.5)

In [138]:
# here we'll only consider the test set from knn split because this set contains the test_size percentage 
# of nearest elements from the target instance
# target_X and target_Y is into test_X and test_Y, we drop the target instance again

new_X_no_tgt= test.drop(target_instance.index)
new_Y_no_tgt= labels_test.drop(index= target_index)

In [140]:
repeat_train= 1
num_fts= len(train.columns)

start= time.time()

# Feature Attribution using Raking - Knn-Local - Remove and Retrain
train_1, test_1, labels_train_1, labels_test_1= knn_train_test_split(new_X_no_tgt,new_Y_no_tgt,
                                                                     target_instance,target_label,
                                                                     test_size=0.2)

acc_all_fts= train_model_get_acc_mean(rf, train_1, test_1, labels_train_1.values.ravel(), 
                                      labels_test_1.values.ravel(), repeat_train)

replace_ft= replace_values(train_1,train_1.columns,num_type='mean',cat_type='median')

acc_no_i, acc_no_ij= remove_and_retrain_v1(rf, replace_ft, train_1, test_1, labels_train_1,
                                           labels_test_1, repeat_train)

end= time.time()
print("--- %s seconds ---" % np.round((end- start), 2))

--- 122.51 seconds ---


In [141]:
print(acc_all_fts)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

0.8888888888888888
-----------------------------
[0.878 0.867 0.911 0.6   0.889 0.889 0.9   0.867 0.833 0.889 0.911 0.9  ]
-----------------------------
[[0.    0.911 0.811 0.922 0.878 0.878 0.878 0.878 0.878 0.878 0.878 0.878]
 [0.911 0.    0.867 0.656 0.889 0.867 0.9   0.867 0.856 0.867 0.889 0.889]
 [0.811 0.867 0.    0.722 0.878 0.878 0.9   0.878 0.878 0.867 0.9   0.9  ]
 [0.922 0.656 0.6   0.    0.722 0.6   0.722 0.6   0.6   0.6   0.722 0.722]
 [0.878 0.867 0.889 0.722 0.    0.9   0.867 0.867 0.878 0.867 0.911 0.922]
 [0.878 0.9   0.911 0.722 0.9   0.    0.922 0.867 0.889 0.9   0.889 0.911]
 [0.878 0.867 0.867 0.722 0.9   0.9   0.    0.878 0.844 0.9   0.878 0.9  ]
 [0.878 0.844 0.878 0.6   0.867 0.867 0.867 0.    0.6   0.867 0.867 0.9  ]
 [0.878 0.856 0.878 0.6   0.867 0.878 0.867 0.578 0.    0.856 0.867 0.9  ]
 [0.878 0.867 0.911 0.6   0.9   0.9   0.856 0.844 0.889 0.    0.878 0.878]
 [0.878 0.9   0.9   0.722 0.889 0.9   0.867 0.911 0.878 0.9   0.    0.922]
 [0.878 0.889 0.9   0.

In [143]:
start= time.time()

# Feature Attribution using Raking - Local - knn and Kfold Remove and Retrain
acc_all_fts, acc_no_i_k, acc_no_ij_k= knnfold_remove_and_retrain(rf,new_X_no_tgt,new_Y_no_tgt,
                                                                target_instance,target_label,numeric_columns, 
                                                                num_type='mean',cat_type='median', 
                                                                test_size= 0.2)

end= time.time()
print("--- %s seconds ---" % np.round((end- start), 2))

--- 629.61 seconds ---


In [144]:
print(acc_all_fts)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i_k, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij_k, decimals=3))

0.8822222222222222
-----------------------------
[0.864 0.867 0.882 0.624 0.889 0.896 0.891 0.873 0.858 0.876 0.884 0.898]
-----------------------------
[[0.    0.893 0.822 0.922 0.871 0.871 0.871 0.871 0.871 0.871 0.871 0.871]
 [0.889 0.    0.896 0.58  0.873 0.873 0.878 0.856 0.856 0.871 0.878 0.882]
 [0.818 0.896 0.    0.673 0.873 0.889 0.893 0.867 0.867 0.869 0.891 0.893]
 [0.922 0.578 0.624 0.    0.647 0.653 0.647 0.624 0.624 0.622 0.649 0.68 ]
 [0.871 0.876 0.878 0.622 0.    0.896 0.878 0.862 0.864 0.873 0.882 0.896]
 [0.871 0.876 0.893 0.7   0.896 0.    0.904 0.869 0.873 0.887 0.893 0.898]
 [0.871 0.878 0.891 0.671 0.869 0.898 0.    0.856 0.864 0.873 0.887 0.896]
 [0.867 0.856 0.871 0.649 0.862 0.867 0.862 0.    0.596 0.862 0.887 0.889]
 [0.871 0.851 0.871 0.649 0.871 0.884 0.873 0.596 0.    0.871 0.882 0.896]
 [0.871 0.856 0.889 0.649 0.864 0.88  0.876 0.862 0.862 0.    0.876 0.849]
 [0.871 0.882 0.893 0.649 0.887 0.898 0.889 0.882 0.887 0.887 0.    0.911]
 [0.871 0.884 0.896 0.

# Misc

In [47]:
eigval, eigvec= eigs(st_matrix3.T, k=1, which='LM')
eigval

array([1.+0.j])

In [48]:
f= open('datasets/FAR_data.txt', 'w')

for i in range(num_fts):
    for j in range(num_fts):
        line= (str(i) + ',' + str(j) + ',' + str(p_matrix1[i][j]) + '\n')
        f.write(line)
        
f.close()